<a href="https://colab.research.google.com/github/xl4357/MSSP6070/blob/main/WeeklyModules/Week09/Week09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import os
import re
import time
from pathlib import Path

In [3]:
keyword = "illustration"

max_images = 15

save_folder = Path("/content/illustrations")
save_folder.mkdir(parents=True, exist_ok=True)

print("Search keyword:", keyword)
print("Save folder:", save_folder)

Search keyword: illustration
Save folder: /content/illustrations


In [4]:
# Wikimedia Commons API
api_url = "https://commons.wikimedia.org/w/api.php"

params = {
    "action": "query",
    "generator": "search",
    "gsrsearch": keyword,
    "gsrnamespace": 6,
    "gsrlimit": max_images,
    "prop": "imageinfo",
    "iiprop": "url|extmetadata",
    "iiurlwidth": 1200,
    "format": "json"
}

headers = {
    "User-Agent": "IllustrationDownloader/1.0 (educational project)"
}

response = requests.get(
    api_url,
    params=params,
    headers=headers,
    timeout=30
)

response.raise_for_status()
data = response.json()

pages = data.get("query", {}).get("pages", {})

print("Images found:", len(pages))

for page in list(pages.values())[:5]:
    print(page["title"])

Images found: 15
File:'One of the wards in the hospital at Scutari'. Wellcome M0007724 - restoration, cropped.jpg
File:Azulejos Parque Eduardo VII-2.jpg
File:Exocoetus volitans.jpg
File:Gear-kegelzahnrad.svg
File:Grant of arms2.jpg


In [5]:
import html
from urllib.parse import urlparse

downloaded = []
failed = []

for page in pages.values():
    title = page["title"]
    image_info = page.get("imageinfo", [{}])[0]

    image_url = image_info.get("thumburl", image_info.get("url"))

    if not image_url:
        failed.append(title)
        continue

    filename = re.sub(r'[\\/*?:"<>|]', "_", title)
    filename = filename.replace("File:", "").strip()

    extension = Path(urlparse(image_url).path).suffix

    if not extension:
        extension = ".jpg"

    file_path = save_folder / f"{filename}{extension}"

    try:
        image_response = requests.get(
            image_url,
            headers=headers,
            timeout=30
        )
        image_response.raise_for_status()

        content_type = image_response.headers.get(
            "Content-Type", ""
        )

        if not content_type.startswith("image/"):
            failed.append(title)
            continue

        file_path.write_bytes(image_response.content)

        # Save source
        metadata = image_info.get("extmetadata", {})

        downloaded.append({
            "title": title,
            "file_path": str(file_path),
            "source_page": image_info.get("descriptionurl", ""),
            "artist": html.unescape(
                re.sub(
                    "<.*?>", "",
                    metadata.get("Artist", {}).get("value", "")
                )
            ),
            "license": metadata.get(
                "LicenseShortName", {}
            ).get("value", "Check source page"),
            "license_url": metadata.get(
                "LicenseUrl", {}
            ).get("value", "")
        })

        print("Downloaded:", filename)

    except Exception as e:
        failed.append(title)
        print("Failed:", title, "|", e)

    # Avoid sending requests too quickly
    time.sleep(0.5)

print("\nDownload complete.")
print("Successful:", len(downloaded))
print("Failed:", len(failed))

Downloaded: File_'One of the wards in the hospital at Scutari'. Wellcome M0007724 - restoration, cropped.jpg
Downloaded: File_Azulejos Parque Eduardo VII-2.jpg
Downloaded: File_Exocoetus volitans.jpg
Downloaded: File_Gear-kegelzahnrad.svg
Downloaded: File_Grant of arms2.jpg
Downloaded: File_Henry Scott Tuke - Beach Study.jpg
Downloaded: File_Illustration Punica granatum2.jpg
Downloaded: File_J. M. Wright - Edward Scriven - Robert Burns - Halloween.JPG
Downloaded: File_Jean-Baptiste Perronneau - Magdaleine Pinceloup de la Grange, née de Parseval.jpg
Downloaded: File_La Madone de São Paulo painting by Alexis Diaz and INTI in São Paulo downtown.jpg
Downloaded: File_Raphael - The Alba Madonna - Google Art Project.jpg
Downloaded: File_Vitrail Chartres 210209 07.jpg
Downloaded: File_William Holman Hunt - The Scapegoat.jpg
Downloaded: File_Yaroslavl ElijahProphetChurchInterior 7302.jpg
Downloaded: File_עץ על אי מלח באמצע ים המלח.jpg

Download complete.
Successful: 15
Failed: 0


In [6]:
import pandas as pd

results_df = pd.DataFrame(downloaded)

results_df.head()

,title,file_path,source_page,artist,license,license_url
0,File:'One of the wards in the hospital at Scut...,/content/illustrations/File_'One of the wards ...,https://commons.wikimedia.org/wiki/File:%27One...,"\nWilliam Simpson (artist, 1823–1899)\nE. Walk...",CC BY 4.0,https://creativecommons.org/licenses/by/4.0
1,File:Azulejos Parque Eduardo VII-2.jpg,/content/illustrations/File_Azulejos Parque Ed...,https://commons.wikimedia.org/wiki/File:Azulej...,Alvesgaspar,CC BY 2.5,https://creativecommons.org/licenses/by/2.5
2,File:Exocoetus volitans.jpg,/content/illustrations/File_Exocoetus volitans...,https://commons.wikimedia.org/wiki/File:Exocoe...,Gervais et Boulart,Public domain,
3,File:Gear-kegelzahnrad.svg,/content/illustrations/File_Gear-kegelzahnrad....,https://commons.wikimedia.org/wiki/File:Gear-k...,Myriam Thyes,Public domain,
4,File:Grant of arms2.jpg,/content/illustrations/File_Grant of arms2.jpg...,https://commons.wikimedia.org/wiki/File:Grant_...,"King Philip II of Spain (commissioner), Anonym...",Public domain,


In [8]:
#save source
results_df.to_csv(
    save_folder / "illustration_sources.csv",
    index=False
)

In [9]:
import shutil

zip_path = shutil.make_archive(
    "/content/illustrations",
    "zip",
    root_dir="/content/illustrations"
)

print("ZIP file created:", zip_path)

ZIP file created: /content/illustrations.zip


In [11]:
from google.colab import files

files.download("/content/illustrations.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>